# Retriever

A retriever is an interface that accepts a natural language query and return a list of Documents (text + metadata).
its more general than a vector store. retrievers can be backed by vector indexes , keyword engines , web APIs , SQL etc.

Using retrievers we can perform 1) Similarity search 2) MMR 3) Scoring etc. statergies to retrive the relevant data from the vector db.Also if you want to get some data from external source such as wikipedia then you can do use of APIs called  API retrievers in retriever to retrieve the info.

## Types of Retriever:
1) Vector (dense) retriever : Converts query to embedding and performs semantic similarity search using vector distance (cosine, dot product, etc.). DB required  -Vector DB (Chroma, FAISS, Pinecone, Weaviate, Milvus)
#
2) Sparse retriever (BM25/lexical) : Performs keyword-based matching using term frequency and relevance scoring.DB type required - Text/Document Index (BM25, Elasticsearch, OpenSearch, Lucene).
#
3) Ensemble / Hybrid Retrieval : Combines results from Dense + Sparse retrieval and reranks/merges them. DB type required -  Both Vector DB + Keyword Index
#
4) Context Compression / Compressor Retriever : Retrieves documents first, then removes irrelevant content/chunks to reduce context size before sending to LLM.Works on top of any retriever (Vector or Sparse)
#
5) MultiQuery Retriever : Generates multiple variations of the user's query using an LLM and retrieves results for all variations.Usually used with Vector DB, but can work with any retriever

# Vector Store Retriever
Documents are embedded and stored in vector store. query is embedded and nearest neighbors are returned.

In [1]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings  # or appropriate import for your setup
from langchain_community.vectorstores import FAISS

# Create 5 Document objects
docs = [
    Document(page_content="Python is a programming language", metadata={"id": 1}),
    Document(page_content="Java is also a programming language", metadata={"id": 2}),
    Document(page_content="Cats sleep often during the day", metadata={"id": 3}),
    Document(page_content="Dogs bark at strangers", metadata={"id": 4}),
    Document(page_content="Birds fly in the sky", metadata={"id": 5}),
]

# Initialize embedding model
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Build vector store
vs = FAISS.from_documents(docs, embedding=emb)

# Create retriever
retriever = vs.as_retriever(search_kwargs={"k": 3}) # k is the number of documents to return

# Query
query = "Which animals are domestic pets?"
out_docs = retriever.invoke(query)

print("Retriever results:")
for d in out_docs:
    print(f"ID: {d.metadata.get('id')}, Content: {d.page_content}")

d:\LangChainP\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Shrinath\AppData\Local\Temp\ipykernel_14204\1215080451.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Retriever results:
ID: 4, Content: Dogs bark at strangers
ID: 5, Content: Birds fly in the sky
ID: 3, Content: Cats sleep often during the day


### Doing the same above thing but using the similarity search instead of retriever. But similarity search is giving irrelevant info as output.

In [2]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings  # or appropriate import for your setup
from langchain_community.vectorstores import FAISS

# Create 5 Document objects
docs = [
    Document(page_content="Python is a programming language", metadata={"id": 1}),
    Document(page_content="Java is also a programming language", metadata={"id": 2}),
    Document(page_content="Cats sleep often during the day", metadata={"id": 3}),
    Document(page_content="Dogs bark at strangers", metadata={"id": 4}),
    Document(page_content="Birds fly in the sky", metadata={"id": 5}),
]

# Initialize embedding model
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Build vector store
vs = FAISS.from_documents(docs, embedding=emb)


# Create retriever
# retriever = vs.as_retriever(search_kwargs={"k": 3})

# Query
query = "Which animals are domestic pets?"
# out_docs = retriever.invoke(query)
out_docs = vs.similarity_search("query", k=3)

print("Retriever results:")
for d in out_docs:
    print(f"ID: {d.metadata.get('id')}, Content: {d.page_content}")

Retriever results:
ID: 1, Content: Python is a programming language
ID: 2, Content: Java is also a programming language
ID: 5, Content: Birds fly in the sky


### Retriving the info using the retriever but using MMR (Maximal Marginal Relevance) statergy. MMR keeps a balance btw Relevance and diversity.

In [3]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings  
from langchain_community.vectorstores import FAISS

# Create 5 Document objects
docs = [
    Document(page_content="Python is a programming language", metadata={"id": 1}),
    Document(page_content="Java is also a programming language", metadata={"id": 2}),
    Document(page_content="Cats sleep often during the day", metadata={"id": 3}),
    Document(page_content="Dogs bark at strangers", metadata={"id": 4}),
    Document(page_content="Birds fly in the sky", metadata={"id": 5}),
]

# Initialize embedding model
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Build vector store
vs = FAISS.from_documents(docs, embedding=emb)

# Create retriever
# here lamnda_mult is the parameter that controls the trade-off between relevance and diversity in the results. A higher value of lambda_mult will result in more diverse results.
retriever = vs.as_retriever(search_type = "mmr", search_kwargs={"k": 3, "fetch_k": 5, "lambda_mult": 0.5})

# Query
query = "Which animals are domestic pets?"
out_docs = retriever.invoke(query)

print("Retriever results:")
for d in out_docs:
    print(f"ID: {d.metadata.get('id')}, Content: {d.page_content}")

Retriever results:
ID: 4, Content: Dogs bark at strangers
ID: 1, Content: Python is a programming language
ID: 3, Content: Cats sleep often during the day


## Sparse / BM25 Retriever
A sparse retriever in LangChain : Matches documents by keyword. Does not use embeddings. Good when exact keyword matter.
#
BM25 Retriever performs keyword-based retrieval by matching query terms against documents. It scores each document using factors such as term frequency (how often a keyword appears), inverse document frequency (how rare the keyword is across all documents), and document length, then returns the highest-scoring relevant documents.It assigns a BM25 value to each documents and rerank them in descending order to give top k values.

In [ ]:
# workflow :
# 1) Build an index of documents : tokenize them , compute the frequencies , inverse document frequencies.
# 2) When a query arrives , parse the query terms
# 3) Compute BM25 score for each document based on the overlap and term statistics
# 4) Return the top k documents with the highest BM25 scores as the results.

# since it works on keywords matched the highest number of keywods that matches in the query will be ranked higher. 

from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# Create 5 Document objects
docs = [
    Document(page_content="The sky is blue during the day", metadata={"id": 1}), # sky, blue
    Document(page_content="At night the stars light up the sky", metadata={"id": 2}), # sky
    Document(page_content="Blue whales are the largest animals on Earth", metadata={"id": 3}), # blue animal
    Document(page_content="Birds can fly high up in the sky", metadata={"id": 4}), # sky
    Document(page_content="Deep sea creatures live in darkness", metadata={"id": 5}), # No matching
]

# Build BM25 retriever
bm25 = BM25Retriever.from_documents(docs, k=3)

# Run query
results = bm25.invoke("sky blue animals")

print("Top 3 results:")
for d in results:
    print(f"ID: {d.metadata.get('id')}, Content: {d.page_content}")

Top 3 results:
ID: 1, Content: The sky is blue during the day
ID: 3, Content: Blue whales are the largest animals on Earth
ID: 4, Content: Birds can fly high up in the sky


## Ensemble / Hybrid Retriever
Use multiple retriever in parallel (sparse + dense etc), combine results to improve recall or precision.

In [12]:
# How it works : 
# 1) Run each retriever separately to get their results
# 2) Each gives list of docs(with or without scores depending on the retriever)
# 3) Combine the results using wights or some other methods (e.g. normalized scores , voting , merging ).


from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# Documents
docs = [
    Document(page_content="AI research is growing fast", metadata={"id": 1}),
    Document(page_content="Machine learning models perform many tasks", metadata={"id": 2}),
    Document(page_content="Cooking and baking are arts", metadata={"id": 3}),
    Document(page_content="Dogs are loyal animals", metadata={"id": 4}),
    Document(page_content="Software engineering is a discipline", metadata={"id": 5}),
]

# BM25 Retriever
bm25 = BM25Retriever.from_documents(docs)
bm25.k = 2

# Vector Retriever
emb = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

vs = FAISS.from_documents(docs, emb)
vector_retriever = vs.as_retriever(
    search_kwargs={"k": 2}
)

query = "what are models in AI"

# Retrieve from both retrievers
bm25_results = bm25.invoke(query)
vector_results = vector_retriever.invoke(query)

# Hybrid Retrieval (manual merge)
combined = bm25_results + vector_results

# Remove duplicates using document id
unique_docs = {}
for doc in combined:
    unique_docs[doc.metadata["id"]] = doc

results = list(unique_docs.values())

print("Hybrid Retrieval Results:\n")

for doc in results:
    print(f"ID {doc.metadata['id']} : {doc.page_content}")

Hybrid Retrieval Results:

ID 1 : AI research is growing fast
ID 2 : Machine learning models perform many tasks


## Contextual Compression Retriever:
A wrapper retriever : first retrieves documents via a base retriever then compresses or filters the content according to the query. Only the parts of documents relevant to the query survive.
###
A Contextual Compression Retriever first retrieves relevant documents using a base retriever (e.g., Vector Retriever or BM25) and then uses an LLM or compressor to remove irrelevant information from those documents. Instead of sending entire chunks to the LLM, it keeps only the parts that are relevant to the user's query. This reduces token usage, improves response quality, and provides more focused context for generation.

In [14]:
# How it works :
# 1)Use a base retriever (vector or sparse) to get candidate documents.
# 2) Use a document compressor to process each retrieved document: exctract ,filter or summarize the parts most relevant to the query.
# 3) Return the compressed versions of the documents as the final results.


from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter

# 5 Document objects
docs = [
    Document(page_content="Python is a programming language used for web dev and data science", metadata={"id":1}),
    Document(page_content="JavaScript runs in the browser, useful for interactive web pages", metadata={"id":2}),
    Document(page_content="C++ is compiled and used for system programming and games", metadata={"id":3}),
    Document(page_content="Rust aims for safety and performance, used in systems software", metadata={"id":4}),
    Document(page_content="Cooking recipes often require precise measurement and timing", metadata={"id":5}),
]

# Base retriever: vector store
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vs = FAISS.from_documents(docs, embedding=emb)
base_retriever = vs.as_retriever(search_kwargs={"k":3})

# Compressor: drop docs whose embedding is too dissimilar
# similarity_threshold is the cosine similarity threshold below which a document will be filtered out. 
# A value of 0.5 means that only documents with a cosine similarity of 0.5 or higher to the query embedding will be retained in the results.
compressor = EmbeddingsFilter(embeddings=emb, similarity_threshold=0.5)

# ContextualCompressionRetriever combines the base retriever and compressor to return compressed results based on relevance to the query.
cc_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

# Query and full output
query = "Which languages are good for system programming?"
out_docs = cc_retriever.invoke(query)

print("Contextual Compression Results:")
for d in out_docs:
    print(f"ID {d.metadata.get('id')}: {d.page_content}")

Contextual Compression Results:
ID 3: C++ is compiled and used for system programming and games
ID 4: Rust aims for safety and performance, used in systems software
ID 1: Python is a programming language used for web dev and data science


## MultiQuery Retriever :
A retriever that uses an LLM to generate multiple reformulations / perspectives of the users query then retrieves for each of them and returns the unique union of all results.
#
Helps cover cases where the query phrasing ,synonyms or perspective might miss relevant documents.


In [15]:
# How it works :
# 1) Recieves users query . Generate multiple queries via LLM (e.g. paraphrasing , query expansion)
# 2) Use a base retriever (vector or sparse) to retrieve documents for each generated query.
# 3) Combine the results (union , weighted scoring , re-ranking) and optionally duplicate removal to produce the final set of retrieved documents.
# 4) Return the final set of retrieved documents as the output.

In [17]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()
# 5 Document objects
docs = [
    Document(page_content="Machine learning includes supervised, unsupervised, and reinforcement learning", metadata={"id":1}),
    Document(page_content="Deep learning uses neural networks with many layers", metadata={"id":2}),
    Document(page_content="Statistics is foundational to ML", metadata={"id":3}),
    Document(page_content="Reinforcement learning deals with agents, rewards, and environments", metadata={"id":4}),
    Document(page_content="Optimization methods like gradient descent are used in training models", metadata={"id":5}),
]

# Build vector store
emb = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vs = FAISS.from_documents(docs, embedding=emb)
base_retriever = vs.as_retriever(search_kwargs={"k":2})

# LLM for query generation
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

# MultiQueryRetriever
multi_q = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    include_original=True
)

# Query
query = "What is reinforcement learning?"
out_docs = multi_q.invoke(query)

print("MultiQuery Retriever Results:")
for d in out_docs:
    print(f"ID {d.metadata.get('id')}: {d.page_content}")

MultiQuery Retriever Results:
ID 4: Reinforcement learning deals with agents, rewards, and environments
ID 1: Machine learning includes supervised, unsupervised, and reinforcement learning
